# Phase 4 — External validation on INbreast

**Question:** a model trained on CBIS-DDSM (scanned film, USA, 1990s) — does it still work on INbreast (digital mammograms, Portugal, 2008–2010) that it has never seen?

The calibration part of Phase 4 needs no GPU: it runs on the Phase 3 predictions already in the repo (`results/calibration/`).
This notebook does the part that needs a GPU:

1. Train the four Phase 3 variants on the **official CBIS-DDSM training patients** (the Phase 3 weights were not saved).
2. Run them on all 410 INbreast images. **Nothing from INbreast is used for training, tuning or calibration.**
3. Report density agreement on INbreast vs CBIS-DDSM, and an exploratory malignancy proxy (BI-RADS 4–6 vs 1–3).

**Before running**
1. Settings → Accelerator → **GPU T4 x2**. Settings → **Internet on**.
2. Add Input → **"CBIS-DDSM: Breast Cancer Image Dataset"** (by *awsaf49*) → Add.
3. Add Input → **"INbreast Dataset"** (by *ramanathansp20*, the one used in Phase 2) → Add.
4. Run the first **two** cells (about 5 minutes). If cell 2 ends with `ALL CHECKS PASSED`, click **Save Version → Save & Run All (Commit)**. The full run takes about **45 minutes** and continues if you close the browser.
5. When it finishes, open the saved version → **Output** → download **`external_results.zip`** and send it to Claude.

In [ ]:
# 1) Get the code and check the machine
import os, subprocess, sys, torch
REPO = "/kaggle/working/repo"
if not os.path.exists(REPO):
    r = subprocess.run(["git", "clone", "--depth", "1", "https://github.com/sweetmAGIciaN7/mammography-multitask-ai.git", REPO],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise SystemExit("Could not download the code. Is Internet ON (Settings -> Internet)?\n" + r.stderr)
print("code:", subprocess.run(["git", "-C", REPO, "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout)
N_GPU = torch.cuda.device_count()
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(N_GPU)] or "NONE")
assert N_GPU > 0, "Turn on the GPU: Settings -> Accelerator -> GPU T4 x2, then run again."
try:
    import pydicom
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pydicom"], check=True)
    import pydicom
print("pydicom", pydicom.__version__)
os.environ["PYTHONPATH"] = f"{REPO}/src"

In [ ]:
# 2) Unit tests + 5-minute smoke run on a tiny subset. If this cell fails, stop and send Claude a screenshot.
!cd /kaggle/working/repo && python -m pytest -q tests 2>&1 | tail -3
!cd /kaggle/working/repo && python -m mammo.experiments.cbis prepare --smoke \
  && python -m mammo.experiments.external train mt_cbam st_dens --smoke \
  && python -m mammo.experiments.external prepare --smoke \
  && python -m mammo.experiments.external predict --smoke \
  && python -m mammo.experiments.external summarise --smoke \
  && echo "ALL CHECKS PASSED"

In [ ]:
# 3) Preprocess CBIS-DDSM (same as Phase 3, ~5-10 min) and all 410 INbreast DICOMs (~2 min).
#    Check the INbreast preview: breasts should face left, fill the frame and look like normal mammograms.
!cd /kaggle/working/repo && python -m mammo.experiments.cbis prepare && python -m mammo.experiments.external prepare
from IPython.display import Image, display
display(Image("/kaggle/working/results/external/inbreast_preview.png"))

In [ ]:
# 4) Train the four variants on the official CBIS-DDSM training split (~15-20 min on T4 x2).
import subprocess, time
plan = {0: ["mt_cbam", "st_dens"], 1: ["mt_plain", "st_path"]} if N_GPU >= 2 else {0: ["mt_cbam", "st_dens", "mt_plain", "st_path"]}
procs = {}
for gpu, variants in plan.items():
    log = open(f"/kaggle/working/train_gpu{gpu}.log", "w")
    cmd = [sys.executable, "-m", "mammo.experiments.external", "train", *variants, "--gpu", str(gpu)]
    procs[gpu] = subprocess.Popen(cmd, cwd=REPO, stdout=log, stderr=subprocess.STDOUT, env=os.environ.copy())
    print(f"GPU {gpu}: {variants}")

def last_line(gpu):
    lines = [l for l in open(f"/kaggle/working/train_gpu{gpu}.log").read().splitlines() if l.strip()]
    return lines[-1] if lines else "(starting)"

t0 = time.time()
while any(p.poll() is None for p in procs.values()):
    time.sleep(120)
    print(f"[{(time.time() - t0) / 60:5.0f} min] " + " | ".join(f"GPU {g}: {last_line(g)}" for g in procs), flush=True)
for gpu, p in procs.items():
    if p.returncode != 0:
        print(open(f"/kaggle/working/train_gpu{gpu}.log").read()[-5000:])
        raise SystemExit(f"GPU {gpu} failed (see above). Send Claude this output.")
print(f"training finished in {(time.time() - t0) / 60:.0f} min")

In [ ]:
# 5) Predict INbreast with every model, then tables + chart
!cd /kaggle/working/repo && python -m mammo.experiments.external predict && python -m mammo.experiments.external summarise
display(Image("/kaggle/working/results/external/external_chart.png"))

In [ ]:
# 6) Pack the results for Claude -> download external_results.zip from Output (/kaggle/working).
#    The model weights stay in this notebook's Output (checkpoints/) so Phase 5 can reuse them.
!cp /kaggle/working/train_gpu*.log /kaggle/working/results/external/ 2>/dev/null; cd /kaggle/working && rm -f external_results.zip \
  && zip -qr external_results.zip results/external -x "*.npy" && ls -lh external_results.zip checkpoints/